# MODELO DE MACHINE LEARNING "LIQUIDITYSENSE" - EVALUACIÓN INICIAL

## 1. Objetivo y alcance

<div style="text-align: justify;">

El presente notebook corresponde a la **etapa de preparación, selección y control de calidad de la información** del proyecto *LiquiditySense*. Su objetivo es transformar la base original de estados financieros bancarios en una base de trabajo estructurada, consistente y adecuada para las etapas posteriores de ingeniería de variables, análisis exploratorio y modelado de Machine Learning.

En esta etapa no se desarrolla todavía el modelo predictivo. El propósito es establecer una base de datos confiable sobre la cual puedan construirse posteriormente los indicadores de liquidez, crecimiento financiero, estructura de fondeo y calidad de cartera, así como la variable objetivo asociada al deterioro futuro de la liquidez.

La preparación de los datos constituye una etapa fundamental dentro de un proyecto de Machine Learning, debido a que errores, inconsistencias o problemas estructurales presentes en la información de entrada pueden propagarse hacia las variables derivadas y afectar posteriormente el entrenamiento, la capacidad predictiva y la interpretación del modelo.

</div>

### Objetivos específicos

<div style="text-align: justify;">

Los principales objetivos de esta etapa son:

</div>

* Seleccionar únicamente las variables necesarias de la base original.
* Reducir el volumen de información procesada mediante una selección previa de variables.
* Definir un período histórico homogéneo para el análisis.
* Identificar duplicados a nivel banco-fecha.
* Evaluar valores faltantes e infinitos.
* Revisar estadísticos descriptivos y posibles valores atípicos.
* Identificar posibles inconsistencias económicas, como valores negativos de efectivo.
* Analizar el comportamiento de variables con información faltante, particularmente `insured_deposits`.
* Construir una base de trabajo ordenada cronológicamente por banco.
* Generar una base procesada que servirá como insumo para la ingeniería de variables.


## 1. Importaciones y configuraciones

<div style="text-align: justify;">

En esta primera etapa se cargan las librerías necesarias para realizar la lectura, transformación, análisis y almacenamiento de la información. Se utiliza **pandas** como herramienta principal para la manipulación de estructuras tabulares, **NumPy** para operaciones numéricas y detección de valores infinitos, y **Path** para gestionar las rutas de archivos y directorios de manera estructurada.

También se configuran las opciones de visualización de pandas para facilitar la inspección de la base durante el proceso de preparación. La visualización de un mayor número de columnas permite realizar revisiones más completas de la estructura de los datos.

La utilización de librerías estándar y ampliamente utilizadas en el ecosistema de Python favorece la reproducibilidad del proceso y permite mantener una separación clara entre las actividades de adquisición, preparación y análisis de los datos.

</div>

**Resultado:** Se cargan correctamente las librerías necesarias y se configura el entorno de trabajo para iniciar el procesamiento de la base.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("✅ Librerías cargadas correctamente")

✅ Librerías cargadas correctamente


## 2. Definición de rutas

<div style="text-align: justify;">

En esta etapa se establecen las rutas correspondientes a la **base de datos original** y al directorio donde se almacenarán los archivos procesados. La separación entre la información original y la información transformada constituye una buena práctica de Data Engineering, ya que permite conservar la fuente de datos sin modificaciones y mantener diferentes etapas del procesamiento de manera trazable.

La base original corresponde al archivo de estados financieros bancarios en formato Stata, mientras que los resultados derivados se almacenan en la estructura de directorios del proyecto *LiquiditySense*.

Adicionalmente, se verifica y crea automáticamente la carpeta de procesamiento en caso de que esta no exista. Esto permite que el notebook pueda ejecutarse de manera reproducible sin depender de la creación manual de directorios.

</div>

**Resultado:** Se establecen las rutas de entrada y salida y se garantiza la existencia del directorio destinado a almacenar las bases procesadas.

In [3]:
# ============================================
# RUTAS
# ============================================

ruta_base = Path(
    r"D:\Edwin\DIPLOMADO INGENIERIA DE DATOS\Machine Learning II\Proyecto\LiquiditySense\data\raw\call-reports-balance-sheets-Jun2026.dta"
)

ruta_procesada = Path("../data/processed")

ruta_procesada.mkdir(parents=True, exist_ok=True)

print("Base original:", ruta_base)
print("Carpeta procesada:", ruta_procesada)

Base original: D:\Edwin\DIPLOMADO INGENIERIA DE DATOS\Machine Learning II\Proyecto\LiquiditySense\data\raw\call-reports-balance-sheets-Jun2026.dta
Carpeta procesada: ..\data\processed


## 3. Variables necesarias

<div style="text-align: justify;">

La base original contiene un número considerable de variables; sin embargo, no todas son necesarias para el objetivo del proyecto. Por esta razón, se realiza una **selección inicial de variables basada en criterios económicos, financieros y metodológicos**.

Las variables seleccionadas permiten representar diferentes dimensiones relevantes para el análisis del riesgo de liquidez. En primer lugar, se incorporan variables de identificación y temporalidad, necesarias para mantener la estructura longitudinal de la información. Posteriormente, se consideran variables relacionadas con activos líquidos, efectivo y valores, así como depósitos y diferentes fuentes de fondeo.

También se incluyen variables relacionadas con la cartera de créditos, su composición y calidad, además de pasivos, patrimonio y deuda subordinada. Esta combinación permite disponer de información suficiente para construir posteriormente indicadores de liquidez, estructura financiera, crecimiento y calidad de cartera.

La selección temprana de variables también permite disminuir el volumen de información que debe ser cargado y procesado, reduciendo el consumo de memoria y mejorando la eficiencia computacional. Esta decisión resulta especialmente relevante debido al tamaño de la base original.

</div>

**Resultado:** Se seleccionan **21 variables** consideradas necesarias para la preparación inicial y para la posterior construcción de variables financieras del proyecto.

In [4]:
variables_base = [
    # Identificación y fecha
    "id_rssd",
    "date",

    # Activos y liquidez
    "assets",
    "cash",
    "securities",

    # Depósitos / fondeo
    "deposits",
    "demand_deposits",
    "time_deposits",
    "brokered_dep",
    "insured_deposits",

    # Cartera de créditos
    "ln_tot",
    "ln_tot_gross",
    "ln_re",
    "ln_ci",
    "ln_cons",
    "ln_agr",
    "ln_rre",

    # Calidad de cartera
    "npl_tot",

    # Pasivos y patrimonio
    "liab_tot",
    "equity",
    "subdebt"
]

print(f"Variables seleccionadas: {len(variables_base)}")

Variables seleccionadas: 21


## 4. Carga de las principales variables

<div style="text-align: justify;">

Una vez definido el conjunto inicial de variables, se procede a cargar únicamente estas columnas desde la base original en formato Stata. Esta estrategia evita cargar las 194 variables disponibles cuando el proyecto requiere solamente un subconjunto de información.

La lectura selectiva constituye una medida de **eficiencia computacional**, debido a que disminuye la cantidad de información transferida desde el archivo hacia la memoria del entorno de Python. Esta decisión es particularmente importante considerando que la base original contiene aproximadamente **2,56 millones de registros y 194 variables**.

La información cargada constituye el punto de partida para las validaciones de estructura, calidad y consistencia económica que se desarrollarán en las siguientes etapas.

</div>

**Resultado:** Se obtiene un DataFrame inicial compuesto exclusivamente por las variables seleccionadas para el proyecto.

In [5]:
df = pd.read_stata(
    ruta_base,
    columns=variables_base
)

print(f"✅ Base cargada")
print(f"Registros: {len(df):,}")
print(f"Variables: {len(df.columns)}")

✅ Base cargada
Registros: 2,557,391
Variables: 21


## 5. Revisión de estructura

<div style="text-align: justify;">

Antes de realizar transformaciones sobre la información, se verifica la estructura del DataFrame mediante la revisión de los tipos de datos, las primeras observaciones y el consumo de memoria.

La revisión de los tipos de datos permite comprobar que las variables hayan sido interpretadas correctamente durante la lectura del archivo Stata. Esto es especialmente importante para variables como `date`, que posteriormente serán utilizadas para establecer el orden temporal y realizar filtros por período.

La inspección de las primeras observaciones permite identificar de manera preliminar la estructura de los registros, mientras que el cálculo del consumo de memoria permite monitorear los recursos utilizados durante el procesamiento.

Este control inicial forma parte de las buenas prácticas de preparación de datos, ya que permite detectar problemas estructurales antes de iniciar transformaciones más complejas.

</div>

**Resultado:** Se verifica la estructura inicial de la información, sus tipos de datos, primeras observaciones y requerimientos de memoria.

In [5]:
print("\nTipos de datos:")
print(df.dtypes)

print("\nPrimeras 5 filas:")
display(df.head())

print("\nMemoria utilizada:")
print(f"{df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


Tipos de datos:
id_rssd                      int32
date                datetime64[ns]
assets                     float64
cash                       float64
securities                 float64
deposits                   float64
demand_deposits            float64
time_deposits              float64
brokered_dep               float64
insured_deposits           float64
ln_tot                     float64
ln_tot_gross               float64
ln_re                      float64
ln_ci                      float64
ln_cons                    float64
ln_agr                     float64
ln_rre                     float64
npl_tot                    float64
liab_tot                   float64
equity                     float64
subdebt                    float64
dtype: object

Primeras 5 filas:


,id_rssd,date,assets,cash,securities,deposits,demand_deposits,time_deposits,brokered_dep,insured_deposits,ln_tot,ln_tot_gross,ln_re,ln_ci,ln_cons,ln_agr,ln_rre,npl_tot,liab_tot,equity,subdebt
0,28,1986-01-01,2779.0,1395.0,0.0,769.0,200.0,165.0,0.0,NaN,23.0,23.0,0.0,0.0,23.0,0.0,NaN,0.0,788.0,1991.0,0.0
1,28,1986-04-01,4322.0,153.0,408.0,2325.0,475.0,570.0,0.0,1320.0,1908.0,1912.0,808.0,284.0,820.0,0.0,NaN,0.0,2345.0,1977.0,0.0
2,28,1986-07-01,5118.0,601.0,407.0,3124.0,561.0,1199.0,0.0,NaN,2978.0,2983.0,1406.0,509.0,1063.0,0.0,NaN,0.0,3150.0,1968.0,0.0
3,28,1986-10-01,5742.0,292.0,611.0,3751.0,496.0,1621.0,0.0,NaN,3422.0,3429.0,1701.0,717.0,995.0,0.0,NaN,0.0,3786.0,1956.0,0.0
4,28,1987-01-01,7737.0,701.0,610.0,5746.0,902.0,2405.0,0.0,NaN,4697.0,4704.0,2451.0,1038.0,1179.0,0.0,NaN,0.0,5795.0,1942.0,0.0



Memoria utilizada:
399.98 MB


## 6. Filtro temporal

<div style="text-align: justify;">

Se define como período de análisis el intervalo comprendido entre **enero de 2000 y 2025**, con el objetivo de trabajar con una ventana histórica homogénea y suficientemente amplia para el desarrollo del modelo.

La utilización de información histórica es especialmente importante en este proyecto debido a que el objetivo consiste en predecir el **deterioro futuro de la liquidez**. Por lo tanto, las observaciones deben conservar su dimensión temporal y permitir posteriormente la construcción de variables rezagadas, variaciones trimestrales y crecimientos interanuales.

Luego de aplicar el filtro temporal, la información queda conformada por **698.184 registros correspondientes a 11.339 instituciones bancarias**, con observaciones comprendidas entre **2000-01-01 y 2025-10-01**.

La definición del período en esta etapa también permite establecer una referencia temporal común para las etapas posteriores de ingeniería de variables y división temporal de los datos en entrenamiento, validación y prueba.

</div>

**Resultado:** La base queda restringida al período 2000–2025, manteniendo **698.184 registros y 11.339 bancos**.

In [6]:
df = df[
    (df["date"].dt.year >= 2000) &
    (df["date"].dt.year <= 2025)
].copy()

print(f"Registros después del filtro temporal: {len(df):,}")
print(f"Bancos: {df['id_rssd'].nunique():,}")
print(f"Fecha mínima: {df['date'].min()}")
print(f"Fecha máxima: {df['date'].max()}")

Registros después del filtro temporal: 698,184
Bancos: 11,339
Fecha mínima: 2000-01-01 00:00:00
Fecha máxima: 2025-10-01 00:00:00


## 7. Control de calidad

<div style="text-align: justify;">

Una vez definido el período de análisis, se realiza un proceso de **control de calidad de los datos (Data Quality Control)**. El objetivo es identificar problemas que puedan afectar la construcción de variables financieras y el posterior entrenamiento del modelo.

En esta etapa se revisan duplicidades, valores faltantes, valores infinitos, estadísticos descriptivos y posibles inconsistencias económicas. La revisión se realiza antes de construir variables derivadas para evitar que problemas presentes en la información original se propaguen hacia las etapas posteriores.

El control de calidad es particularmente importante en información financiera, debido a que un valor incorrecto o una estructura temporal inconsistente puede generar indicadores financieros artificiales y afectar la capacidad predictiva del modelo.

</div>

### 7.1 Identificación de duplicados

<div style="text-align: justify;">

Se verifica la existencia de registros duplicados utilizando como llave la combinación de `id_rssd` y `date`. Esta combinación representa la identificación de una institución bancaria en un período determinado.

La validación es necesaria porque posteriormente se realizarán operaciones temporales agrupadas por institución bancaria. Si existieran múltiples registros para un mismo banco y fecha, podrían generarse errores en los cálculos de variaciones trimestrales, crecimientos interanuales y variables rezagadas.

Por lo tanto, se espera que cada combinación banco-fecha corresponda a una única observación dentro de la base.

</div>

**Resultado:** Se verifica la unicidad de la combinación `id_rssd`–`date`, sin identificarse registros duplicados.

In [7]:
# ============================================
# CONTROL DE DUPLICADOS
# ============================================

duplicados = df.duplicated(
    subset=["id_rssd", "date"]
).sum()

print(f"Registros duplicados banco-fecha: {duplicados:,}")

Registros duplicados banco-fecha: 0


### 7.2 Valores faltantes

<div style="text-align: justify;">

Se cuantifican los valores faltantes de cada variable tanto en términos absolutos como porcentuales. Esta revisión permite conocer la disponibilidad de información y detectar variables que puedan presentar patrones importantes de ausencia.

En esta etapa los valores faltantes **no se eliminan automáticamente**. Esta decisión responde a que un valor ausente en información bancaria puede tener diferentes causas: ausencia de reporte, característica estructural de una institución, cambio en la forma de reporte o verdadera falta de información.

Por este motivo, antes de decidir un tratamiento estadístico, se analiza el comportamiento de los valores faltantes y su relación con las características financieras de los registros afectados.

Este enfoque evita perder información de manera innecesaria y permite que el tratamiento posterior de los faltantes se realice de forma metodológicamente fundamentada.

</div>

**Resultado:** Se genera un resumen de valores faltantes por variable y se identifican los registros que requieren análisis posterior.

In [8]:
# ============================================
# VALORES FALTANTES
# ============================================

resumen_nulos = pd.DataFrame({
    "Variable": df.columns,
    "Nulos": df.isna().sum().values,
    "% Nulos": (
        df.isna().mean() * 100
    ).round(2).values
})

print(resumen_nulos.to_string(index=False))

        Variable  Nulos  % Nulos
         id_rssd      0     0.00
            date      0     0.00
          assets      3     0.00
            cash      3     0.00
      securities      3     0.00
        deposits      3     0.00
 demand_deposits      3     0.00
   time_deposits      3     0.00
    brokered_dep      3     0.00
insured_deposits   2922     0.42
          ln_tot      3     0.00
    ln_tot_gross      3     0.00
           ln_re      3     0.00
           ln_ci      3     0.00
         ln_cons      3     0.00
          ln_agr      3     0.00
          ln_rre      3     0.00
         npl_tot      3     0.00
        liab_tot      3     0.00
          equity      3     0.00
         subdebt      3     0.00


### 7.3 Valores infinitos

<div style="text-align: justify;">

Se realiza una revisión de valores infinitos (`inf` y `-inf`) en las variables numéricas de la base. Los valores infinitos pueden aparecer como consecuencia de operaciones matemáticas en las que intervienen divisiones por cero o denominadores extremadamente pequeños.

Aunque en esta etapa todavía no se construyen los principales indicadores financieros, esta validación establece una línea base de calidad antes de realizar las transformaciones correspondientes a la ingeniería de variables.

La identificación temprana de estos valores es importante debido a que los algoritmos de transformación, escalamiento y Machine Learning generalmente requieren valores numéricos finitos.

</div>

**Resultado:** Se realiza la revisión de valores infinitos en las variables numéricas de la base antes de iniciar la generación de variables derivadas.

In [9]:
# ============================================
# VALORES INFINITOS
# ============================================

infinitos = pd.DataFrame({
    "Variable": df.columns,
    "Infinitos": [
        np.isinf(df[col]).sum()
        if pd.api.types.is_numeric_dtype(df[col])
        else 0
        for col in df.columns
    ]
})

print(infinitos.to_string(index=False))

        Variable  Infinitos
         id_rssd          0
            date          0
          assets          0
            cash          0
      securities          0
        deposits          0
 demand_deposits          0
   time_deposits          0
    brokered_dep          0
insured_deposits          0
          ln_tot          0
    ln_tot_gross          0
           ln_re          0
           ln_ci          0
         ln_cons          0
          ln_agr          0
          ln_rre          0
         npl_tot          0
        liab_tot          0
          equity          0
         subdebt          0


### 7.4 Estadísticos preliminares

<div style="text-align: justify;">

Se calculan estadísticos descriptivos para las variables numéricas utilizando diferentes percentiles de la distribución. Esta revisión permite conocer la escala, tendencia central y dispersión de las principales variables financieras.

Se consideran los percentiles 1%, 5%, 25%, 50%, 75%, 95% y 99%, además de los estadísticos tradicionales. La utilización de percentiles permite observar el comportamiento de las variables en diferentes puntos de su distribución e identificar posibles valores extremos.

En información financiera, la presencia de valores extremos no implica necesariamente que exista un error. Un valor elevado o reducido puede representar una situación económica real de una institución bancaria. Por esta razón, los estadísticos descriptivos se utilizan inicialmente como herramienta de diagnóstico y no como criterio automático de eliminación.

Los resultados obtenidos en esta etapa servirán posteriormente como referencia para la construcción y revisión de los indicadores financieros.

</div>

**Resultado:** Se obtiene una caracterización estadística inicial de las variables seleccionadas y de su distribución.

In [10]:
# ============================================
# ESTADÍSTICOS DESCRIPTIVOS
# ============================================

estadisticos = df.describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

display(estadisticos)

,count,mean,min,1%,5%,25%,50%,75%,95%,99%,max,std
id_rssd,698184.0,992519.330857,37.0,12955.0,64646.0,326979.0,653947.0,971379.0,3301641.0,3655933.0,6009630.0,1018360.493132
date,698184,2011-04-20 05:38:43.275239680,2000-01-01 00:00:00,2000-01-01 00:00:00,2000-10-01 00:00:00,2005-01-01 00:00:00,2010-10-01 00:00:00,2017-04-01 00:00:00,2024-01-01 00:00:00,2025-07-01 00:00:00,2025-10-01 00:00:00,NaN
assets,698181.0,2214073.16378,0.0,9270.8,22840.0,68871.0,152623.0,378547.0,2329851.0,21027840.0,3813431000.0,41657697.735308
cash,698181.0,220957.127291,-178.0,336.0,931.0,3250.0,8244.0,22437.0,141867.0,1170641.0,760259000.0,5843768.75789
securities,698181.0,463036.389259,0.0,0.0,963.0,10247.0,27752.0,74742.0,438300.0,3787905.2,941745000.0,9424512.85286
deposits,698181.0,1660488.178104,0.0,25.6,17214.0,56826.0,126706.0,310840.0,1814656.0,15624477.2,2697842000.0,31070080.368242
demand_deposits,698181.0,282460.588028,0.0,0.0,1510.0,7447.0,17895.0,43312.0,247755.0,1870960.8,1233311000.0,7589642.208546
time_deposits,698181.0,273800.897201,0.0,0.0,5582.0,21348.0,46658.0,108491.0,516985.0,3127220.8,303937000.0,2953140.128555
brokered_dep,698181.0,103293.547417,0.0,0.0,0.0,0.0,0.0,3400.0,84941.0,1002544.8,139426000.0,1843309.612732
insured_deposits,695262.0,740681.90458,0.0,3.0,12230.05,38544.0,83067.0,194210.5,999531.8,7703074.02,901146000.0,11854413.733932


### 7.5 Validación complementaria de duplicados

<div style="text-align: justify;">

Se realiza nuevamente la validación de duplicados utilizando la combinación `id_rssd` y `date`. Esta segunda comprobación permite confirmar que la estructura de la información continúa manteniendo una observación única por institución y período después de las revisiones realizadas.

La validación es relevante porque la base será utilizada como una estructura longitudinal. La existencia de duplicados podría alterar la secuencia temporal y generar resultados incorrectos al utilizar funciones de desplazamiento como `shift()` durante la etapa de ingeniería de variables.

</div>

**Resultado:** Se confirma la ausencia de registros duplicados a nivel banco-fecha.

In [11]:
duplicados = df.duplicated(
    subset=["id_rssd", "date"]
).sum()

print(f"Registros duplicados banco-fecha: {duplicados:,}")

Registros duplicados banco-fecha: 0


### 7.6 Identificación de registros con valores nulos

<div style="text-align: justify;">

Además del análisis agregado de valores faltantes por variable, se identifican los registros que presentan al menos un valor nulo. Esta revisión permite analizar los faltantes desde una perspectiva individual y determinar si se concentran en determinadas instituciones o períodos.

El análisis a nivel de registro resulta importante porque posteriormente será necesario decidir cómo tratar los valores faltantes. La estrategia adoptada no debe basarse únicamente en la cantidad de valores ausentes, sino también en su naturaleza y comportamiento.

De esta manera, se evita eliminar automáticamente observaciones potencialmente útiles y se mantiene la posibilidad de incorporar posteriormente mecanismos de imputación o indicadores de ausencia cuando estos tengan significado estadístico o económico.

</div>

**Resultado:** Se identifican los registros que contienen al menos un valor faltante para su análisis y tratamiento posterior.

In [12]:
# ============================================
# IDENTIFICAR REGISTROS CON NULOS
# ============================================

registros_nulos = df[
    df[variables_base].isna().any(axis=1)
].copy()

print(f"Registros con al menos un nulo: {len(registros_nulos):,}")

print(registros_nulos)

Registros con al menos un nulo: 2,922
         id_rssd       date     assets       cash  securities  deposits  \
127631     52719 2011-07-01  3676961.0  2967847.0    205302.0       0.0   
127632     52719 2011-10-01  3551202.0  2881842.0    205302.0       0.0   
127633     52719 2012-01-01  3388853.0  2730282.0    205302.0       0.0   
127634     52719 2012-04-01  3343419.0  2691264.0    205302.0       0.0   
127635     52719 2012-07-01  4242910.0  3541147.0    205302.0       0.0   
...          ...        ...        ...        ...         ...       ...   
2557381  6009630 2024-10-01    26027.0      252.0         0.0       0.0   
2557382  6009630 2025-01-01    28474.0      263.0         0.0       0.0   
2557383  6009630 2025-04-01    29775.0      260.0         0.0       0.0   
2557384  6009630 2025-07-01    31987.0      524.0         0.0       0.0   
2557385  6009630 2025-10-01    29119.0      254.0         0.0       0.0   

         demand_deposits  time_deposits  brokered_dep  insure

### 7.7 Revisión de valores negativos de `cash`

<div style="text-align: justify;">

Se realiza una revisión específica de los registros en los que la variable `cash` presenta valores negativos. Desde una perspectiva económica, el efectivo y equivalentes de efectivo normalmente representan activos y, por tanto, un valor negativo requiere una revisión adicional antes de ser utilizado en la construcción de indicadores de liquidez.

Sin embargo, un valor negativo no debe eliminarse automáticamente únicamente por no coincidir con la expectativa económica inicial. Puede estar relacionado con criterios específicos de reporte, reclasificaciones contables o situaciones particulares de la información financiera.

Por esta razón, se identifican los registros afectados y se contrastan con variables relacionadas como activos, efectivo, depósitos y otras partidas financieras.

</div>

**Resultado:** Se identifican los registros con `cash < 0` para realizar una revisión económica y determinar posteriormente el tratamiento adecuado.

In [13]:
cash_negativo = df[df["cash"] < 0][
    ["id_rssd", "date", "assets", "cash", "deposits"]
]

print(f"Registros con cash negativo: {len(cash_negativo):,}")

display(cash_negativo.head(20))

Registros con cash negativo: 2


,id_rssd,date,assets,cash,deposits
2476110,2802945,2005-01-01,73991.0,-178.0,58036.0
2476111,2802945,2005-04-01,95591.0,-38.0,72031.0


## 8. Revisión de `insured_deposits`

<div style="text-align: justify;">

Se realiza un análisis específico de la variable `insured_deposits`, debido a que presenta valores faltantes que requieren determinar si corresponden a un problema de calidad o a una característica estructural de la información reportada.

El análisis considera el número de registros afectados, las instituciones involucradas y el período en el que se presentan los valores faltantes. Adicionalmente, se revisa el comportamiento de variables relacionadas, principalmente los depósitos y la cartera de créditos, con el objetivo de identificar posibles patrones que permitan explicar la ausencia de información.

Este análisis es importante porque `insured_deposits` puede ser utilizada posteriormente para construir indicadores relacionados con la estructura y estabilidad del fondeo. Por tanto, comprender la naturaleza de sus valores faltantes permite definir un tratamiento más adecuado durante la ingeniería de variables y el preprocesamiento del modelo.

</div>

### 8.1 Análisis de los registros afectados

<div style="text-align: justify;">

Se identifican los registros con valores nulos en `insured_deposits` y se determina el número de instituciones bancarias afectadas. También se revisa el período temporal en el que se concentran estos registros.

Posteriormente, se analizan estadísticos descriptivos de `deposits` y `ln_tot` para comparar las características financieras de las observaciones afectadas.

</div>

**Resultado:** Se caracteriza la distribución temporal y financiera de los registros con información faltante en `insured_deposits`.

In [14]:
# ============================================
# ANÁLISIS DE INSURED DEPOSITS
# ============================================

nulos_insured = df[df["insured_deposits"].isna()].copy()

print("Registros con insured_deposits nulo:", len(nulos_insured))
print("Bancos afectados:", nulos_insured["id_rssd"].nunique())

print("\nFecha mínima:")
print(nulos_insured["date"].min())

print("\nFecha máxima:")
print(nulos_insured["date"].max())

print("\nDepósitos de los registros afectados:")
print(nulos_insured["deposits"].describe())

print("\nCréditos de los registros afectados:")
print(nulos_insured["ln_tot"].describe())

Registros con insured_deposits nulo: 2922
Bancos afectados: 85

Fecha mínima:
2001-07-01 00:00:00

Fecha máxima:
2025-10-01 00:00:00

Depósitos de los registros afectados:
count      2919.000000
mean        121.912641
std        6574.671088
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max      355215.000000
Name: deposits, dtype: float64

Créditos de los registros afectados:
count      2919.000000
mean       1017.787256
std       16550.256521
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max      271695.000000
Name: ln_tot, dtype: float64


### 8.2 Análisis de `insured_deposits` frente a `deposits`

<div style="text-align: justify;">

Se construye una tabla de contingencia que relaciona la condición de valor faltante de `insured_deposits` con la condición de `deposits` igual a cero.

Esta comparación permite determinar si los valores faltantes de los depósitos asegurados se encuentran asociados a instituciones o períodos en los que no existen depósitos reportados. Si se observa una relación sistemática entre ambas condiciones, el patrón de ausencia podría corresponder a una característica estructural de la información y no necesariamente a un error de registro.

El análisis permite fundamentar posteriormente el tratamiento de esta variable evitando realizar imputaciones arbitrarias que puedan introducir información artificial.

</div>

**Resultado:** Se utiliza la relación entre ambas variables como evidencia para evaluar la naturaleza de los valores faltantes de `insured_deposits`.

In [15]:
# ============================================
# INSURED DEPOSITS VS DEPOSITS
# ============================================

pd.crosstab(
    df["insured_deposits"].isna(),
    df["deposits"].eq(0)
)

deposits,False,True
insured_deposits,,
False,691255,4007
True,25,2897


### 8.3 Revisión ampliada de valores negativos de `cash`

<div style="text-align: justify;">

Se amplía la revisión de los registros con `cash` negativo incorporando variables financieras adicionales como `securities`, `deposits`, `ln_tot`, `liab_tot` y `equity`.

El objetivo es evaluar estos registros dentro de su contexto financiero y evitar tomar decisiones de limpieza basadas únicamente en una variable. Una observación que presenta un valor negativo de efectivo debe analizarse conjuntamente con la estructura del activo, pasivo, depósitos y patrimonio de la institución.

Este procedimiento permite diferenciar entre posibles inconsistencias de información y situaciones que puedan tener una explicación asociada a la estructura de reporte financiero.

</div>

**Resultado:** Se obtiene una vista ampliada de los registros con `cash` negativo para sustentar su tratamiento en las siguientes etapas.

In [16]:
# ============================================
# CASH NEGATIVO
# ============================================

cash_negativo = df[
    df["cash"] < 0
][[
    "id_rssd",
    "date",
    "assets",
    "cash",
    "securities",
    "deposits",
    "ln_tot",
    "liab_tot",
    "equity"
]]

print(cash_negativo)

         id_rssd       date   assets   cash  securities  deposits   ln_tot  \
2476110  2802945 2005-01-01  73991.0 -178.0      4627.0   58036.0  66614.0   
2476111  2802945 2005-04-01  95591.0  -38.0      4663.0   72031.0  88513.0   

         liab_tot  equity  
2476110   68102.0  5889.0  
2476111   87108.0  8483.0  


## 9. Limpieza inicial y construcción de la base de trabajo

<div style="text-align: justify;">

Una vez completadas las revisiones iniciales, se construye la **base de trabajo** que será utilizada en las siguientes etapas del proyecto. Esta base contiene únicamente las variables consideradas necesarias para el análisis y conserva la estructura longitudinal de la información.

La selección de las variables en esta etapa permite establecer un conjunto controlado de información que será posteriormente complementado mediante la ingeniería de variables. A partir de esta base se construirán indicadores financieros, tasas de crecimiento, variables rezagadas, indicadores de valores faltantes y finalmente la variable objetivo.

</div>

### 9.1 Ordenamiento temporal

<div style="text-align: justify;">

Los registros son ordenados primero por `id_rssd` y posteriormente por `date`. Este ordenamiento es fundamental para garantizar que cada institución bancaria mantenga sus observaciones en secuencia cronológica.

La correcta organización temporal constituye un requisito esencial para las operaciones de ingeniería de variables que utilizarán información histórica. Por ejemplo, para calcular variaciones trimestrales o interanuales será necesario comparar cada observación con períodos anteriores de la misma institución.

Mantener esta estructura permite reducir el riesgo de utilizar accidentalmente información futura para explicar una observación pasada, contribuyendo al control de **data leakage** durante las etapas posteriores del proyecto.

</div>

**Resultado:** Se obtiene una base de trabajo organizada por institución bancaria y fecha, preparada para la construcción de variables históricas y financieras.

### 9.2 Validación de la base de trabajo

<div style="text-align: justify;">

Después de construir y ordenar la base de trabajo, se realiza una validación final de su estructura. Se verifica el número de registros, cantidad de variables, número de instituciones, período de análisis, duplicados banco-fecha y consumo de memoria.

Estas comprobaciones permiten confirmar que la selección y ordenamiento de la información no hayan alterado la estructura esperada de la base.

La validación final constituye un punto de control antes de almacenar la información y avanzar hacia la etapa de ingeniería de variables.

</div>

**Resultado:** La base de trabajo mantiene **698.184 registros**, correspondientes a **11.339 instituciones**, con información comprendida entre **2000-01-01 y 2025-10-01**, y sin duplicados a nivel banco-fecha.

In [17]:
# ============================================
# PASO 9 — CREAR BASE DE TRABAJO
# ============================================

variables_analisis = [
    "id_rssd",
    "date",
    "assets",
    "cash",
    "securities",
    "deposits",
    "demand_deposits",
    "time_deposits",
    "brokered_dep",
    "insured_deposits",
    "ln_tot",
    "ln_tot_gross",
    "ln_re",
    "ln_ci",
    "ln_cons",
    "ln_agr",
    "ln_rre",
    "npl_tot",
    "liab_tot",
    "equity",
    "subdebt"
]

# Crear base de trabajo
base = df[variables_analisis].copy()

# Ordenar por banco y fecha
base = (
    base
    .sort_values(["id_rssd", "date"])
    .reset_index(drop=True)
)

# ============================================
# VALIDACIÓN
# ============================================

print("============================================")
print("BASE DE TRABAJO")
print("============================================")

print(f"Registros: {len(base):,}")
print(f"Variables: {len(base.columns)}")
print(f"Bancos: {base['id_rssd'].nunique():,}")
print(f"Fecha mínima: {base['date'].min()}")
print(f"Fecha máxima: {base['date'].max()}")

print("\nDuplicados banco-fecha:")
print(
    base.duplicated(
        subset=["id_rssd", "date"]
    ).sum()
)

print("\nMemoria utilizada:")
print(
    f"{base.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

BASE DE TRABAJO
Registros: 698,184
Variables: 21
Bancos: 11,339
Fecha mínima: 2000-01-01 00:00:00
Fecha máxima: 2025-10-01 00:00:00

Duplicados banco-fecha:
0

Memoria utilizada:
109.20 MB


## 10. Guardar base procesada

<div style="text-align: justify;">

Una vez finalizado el proceso de preparación y validación, la base de trabajo se almacena en formato **CSV** dentro del directorio `data/processed` del proyecto.

El almacenamiento de la base procesada permite separar la etapa de preparación de las etapas posteriores del proyecto y facilita la reproducibilidad del flujo de trabajo. De esta manera, los notebooks siguientes pueden utilizar una base ya estructurada sin necesidad de volver a leer y procesar la fuente original de varios gigabytes.

El archivo generado constituye un **punto de control del pipeline de datos**, permitiendo conservar el resultado de esta etapa y mantener la trazabilidad entre la información original y las transformaciones realizadas.

</div>

### Archivo generado

<div style="text-align: justify;">

La base se almacena como `base_liquidez.csv` dentro de `data/processed`. Este archivo constituye el insumo para el siguiente notebook, correspondiente a la etapa de **ingeniería de variables**, donde se construirán los indicadores financieros y las variables necesarias para el modelo predictivo.

</div>

**Resultado:** Base de trabajo procesada y almacenada correctamente en formato CSV.

In [19]:
# ============================================

# PASO 10 — GUARDAR BASE LIQUIDEZ EN PARQUET

# ============================================

from pathlib import Path

ruta_procesada = Path("../data/processed")

ruta_procesada.mkdir(
parents=True,
exist_ok=True
)

archivo_base = ruta_procesada / "base_liquidez.parquet"

base.to_parquet(
archivo_base,
engine="pyarrow",
index=False
)

print("✅ Base guardada correctamente")
print(f"Ruta: {archivo_base}")
print(f"Registros: {len(base):,}")
print(f"Variables: {len(base.columns):,}")
print(f"Tamaño: {archivo_base.stat().st_size / (1024**2):.2f} MB")



✅ Base guardada correctamente
Ruta: ..\data\processed\base_liquidez.parquet
Registros: 698,184
Variables: 21
Tamaño: 48.03 MB


In [20]:
# ============================================

# PASO 11 — VALIDAR BASE GUARDADA

# ============================================

base_validacion = pd.read_parquet(
archivo_base
)

print("============================================")
print("VALIDACIÓN DE BASE LIQUIDEZ")
print("============================================")

print(f"Registros: {len(base_validacion):,}")
print(f"Variables: {len(base_validacion.columns)}")
print(f"Bancos: {base_validacion['id_rssd'].nunique():,}")
print(f"Fecha mínima: {base_validacion['date'].min()}")
print(f"Fecha máxima: {base_validacion['date'].max()}")

print("\nDuplicados banco-fecha:")

duplicados = base_validacion.duplicated(
subset=["id_rssd", "date"]
).sum()

print(duplicados)

print("\nPrimeras 5 filas:")
display(base_validacion.head())


VALIDACIÓN DE BASE LIQUIDEZ
Registros: 698,184
Variables: 21
Bancos: 11,339
Fecha mínima: 2000-01-01 00:00:00
Fecha máxima: 2025-10-01 00:00:00

Duplicados banco-fecha:
0

Primeras 5 filas:


,id_rssd,date,assets,cash,securities,deposits,demand_deposits,time_deposits,brokered_dep,insured_deposits,ln_tot,ln_tot_gross,ln_re,ln_ci,ln_cons,ln_agr,ln_rre,npl_tot,liab_tot,equity,subdebt
0,37,2000-01-01,64088.0,1436.0,20637.0,52987.0,6446.0,30081.0,0.0,40256.0,36620.0,36620.0,24788.0,790.0,9556.0,37.0,18755.0,360.0,53379.0,10709.0,0.0
1,37,2000-04-01,62982.0,2014.0,20939.0,51772.0,6437.0,29852.0,0.0,40011.0,38458.0,38458.0,25541.0,866.0,10129.0,18.0,19705.0,597.0,52137.0,10845.0,0.0
2,37,2000-07-01,63862.0,1386.0,20395.0,51648.0,6984.0,30208.0,0.0,40156.0,41145.0,41145.0,26315.0,918.0,10753.0,31.0,20677.0,702.0,52773.0,11089.0,0.0
3,37,2000-10-01,62558.0,1465.0,20312.0,51071.0,6461.0,30060.0,0.0,44780.0,38235.0,38235.0,26261.0,1054.0,10458.0,79.0,20767.0,483.0,51354.0,11204.0,0.0
4,37,2001-01-01,65486.0,1729.0,18439.0,53715.0,7024.0,31133.0,0.0,40270.0,40507.0,40507.0,27636.0,711.0,10534.0,106.0,21558.0,426.0,54062.0,11424.0,0.0


# Conclusión de la etapa de preparación

<div style="text-align: justify;">

La etapa de preparación permitió transformar la información original de estados financieros bancarios en una base de trabajo estructurada para el desarrollo del proyecto *LiquiditySense*. El proceso incluyó la selección de variables relevantes, definición del período histórico, revisión de la estructura de los datos, controles de duplicidad, análisis de valores faltantes e infinitos y revisión de posibles inconsistencias financieras.

Como resultado, se dispone de una base longitudinal con **698.184 registros correspondientes a 11.339 instituciones bancarias**, ordenada por institución y fecha y sin duplicados banco-fecha. Esta estructura resulta fundamental para desarrollar posteriormente variables basadas en la evolución histórica de cada banco.

Es importante destacar que en esta etapa no se realizan eliminaciones indiscriminadas de información. Los valores faltantes y posibles inconsistencias económicas son primero identificados y analizados, dejando su tratamiento específico para las etapas en las que su naturaleza y efecto sobre las variables derivadas puedan evaluarse adecuadamente.

La base generada constituye el punto de partida para la siguiente fase del proyecto: **Ingeniería de Variables**, donde se construirán los indicadores de liquidez, estructura de fondeo, crecimiento financiero, calidad de cartera, indicadores de ausencia estructural y la variable objetivo utilizada para entrenar el modelo de Machine Learning.

</div>